# Week 3 (together), Modelling: logistic regression, built by the room

**This notebook is blank on purpose.** Today's group work is the modelling, and the modelling
is a sequence of *choices*: what the two piles are, what counts as a feature, what score you
would believe, and which of several models you keep. If the code were already here you would
read the choices instead of making them.

**What your group does today.** Pick two labelled piles of text, fit a logistic regression on
them, read the weights it learned, try to break it, then change one modelling decision and fit
it again. Bring back a number, two word lists, and one caveat.

**How to work.** Threes, one screen. Rotate the three jobs every station:

- **Driver** types (and prompts the AI). Never types a line no one has read aloud.
- **Reader** says what each cell will do *before* it runs, then whether it did.
- **Skeptic** asks the awkward question: is that score better than guessing? Would that word
  survive on someone else's data?

Only the setup at the top is written for you. The worked version of this pipeline is
`week03_classification.ipynb`, so if you get stuck, ask the AI first, then peek there.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.

In [ ]:
# If an import fails: re-run this cell; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, models, and charts are
# still there next week. (Outside Colab - e.g. the offline test harness - this falls
# back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print("imports ok")

## Setup (written for you): two labelled piles of text

A classifier needs examples that someone has already sorted. `load_pair()` below hands your
group a DataFrame with two columns, `text` and `label`, and nothing else. Collecting the
corpus is Week 4's subject; today it is plumbing.

**Your one setup choice: which two piles.** Set `PAIR` in the next cell.

- `("sandiego", "SanDiegan")` — two subreddits about one city, the hard default. Same beaches,
  same rents, largely the same words.
- Any two subreddit names you actually know: `("coffee", "espresso")`, `("AskMen", "AskWomen")`,
  `("nba", "soccer")`. The archive covers essentially all of Reddit; comments are pulled live
  and used for analysis only, never redistributed.
- `("shelley", "stoker")` — sentences from *Frankenstein* and *Dracula*, no network needed.
  This is also the automatic fallback when the archive is slow or a whole classroom hits it
  at once, so a failed pull never costs you the session.

An easy pair (two unrelated topics) gets you 95 percent accuracy and a boring reading. A hard
pair gets you 60 percent and a real discussion. Pick deliberately, and say which you picked
when you report back.

In [ ]:
# --- Provided loader. Read it, don't rewrite it. ------------------------------
PAIR = ("sandiego", "SanDiegan")   # <-- your group's two piles (see above)
N_PER_SIDE = 400                   # rows per pile; 400 trains in a couple of seconds

def load_reddit(a, b, n=N_PER_SIDE):
    """Two subreddits via the Arctic Shift archive API (analyze-only).
    Retries politely: a whole classroom at once gets rate-limited."""
    import requests, time
    def pull(sub, tries=2):
        for attempt in range(tries):
            got, before, pages = [], None, 0
            try:
                while len(got) < n and pages < 10:
                    params = {"subreddit": sub, "limit": 100, "fields": "body,created_utc"}
                    if before: params["before"] = before
                    resp = requests.get("https://arctic-shift.photon-reddit.com/api/comments/search",
                                        params=params, timeout=30)
                    resp.raise_for_status()
                    rows = resp.json()["data"]
                    pages += 1
                    if not rows: break
                    before = int(min(r["created_utc"] for r in rows))
                    got += [r["body"] for r in rows if isinstance(r.get("body"), str)
                            and 80 < len(r["body"]) < 800
                            and "[removed]" not in r["body"] and "[deleted]" not in r["body"]
                            and "moderator" not in r["body"].lower()
                            and "has been removed" not in r["body"].lower()]
                if got:
                    return got[:n]
            except Exception as e:
                print(f"r/{sub}: {type(e).__name__} (attempt {attempt + 1}/{tries}), waiting...")
            time.sleep(3 * (attempt + 1))
        return []
    a_txt, b_txt = pull(a), pull(b)
    if not (a_txt and b_txt): raise ValueError("empty pull")
    return pd.DataFrame({"text": a_txt + b_txt, "label": [a] * len(a_txt) + [b] * len(b_txt)})

def load_novelists(n=N_PER_SIDE):
    """Shelley against Stoker: the repo's snapshots first, Gutenberg if they aren't there."""
    def sents(text):
        return [s.strip().replace("\n", " ") for s in re.split(r"(?<=[.!?])\s+", text)
                if 40 < len(s) < 180][:n]
    def read_one(fname, url):
        for base in ("data/texts", "notebooks/data/texts", "../notebooks/data/texts"):
            path = os.path.join(base, fname)
            if os.path.exists(path):
                return open(path, encoding="utf-8", errors="ignore").read()
        import requests
        raw = requests.get(url, timeout=30).text.replace("\r\n", "\n")
        body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
        return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]
    a = sents(read_one("frankenstein.txt", "https://www.gutenberg.org/cache/epub/84/pg84.txt"))
    b = sents(read_one("dracula.txt", "https://www.gutenberg.org/cache/epub/345/pg345.txt"))
    return pd.DataFrame({"text": a + b, "label": ["shelley"] * len(a) + ["stoker"] * len(b)})

def load_pair(pair=PAIR):
    """Returns (df, label_a, label_b). Falls back to the novelists if the archive is down."""
    a, b = pair
    if {a, b} == {"shelley", "stoker"}:
        return load_novelists(), "shelley", "stoker"
    try:
        return load_reddit(a, b), a, b
    except Exception as e:
        print("archive unavailable:", type(e).__name__, "- falling back to the novelists")
        return load_novelists(), "shelley", "stoker"

df, LABEL_A, LABEL_B = load_pair()
print(df["label"].value_counts().to_string())
df.sample(4, random_state=1)

---

## Station 1 · The question, and your prediction

Before any modelling. One sentence: what would it *mean* if a machine could tell these two
piles apart, and what would it mean if it couldn't?

Then predict, out loud and in writing: what accuracy do you expect, and name three words you
think will give each side away. Writing the prediction down is what makes the result
informative later; a prediction made after the fact is not a prediction.

In [ ]:
# OUR QUESTION: ...one sentence...
# OUR PREDICTION: accuracy about ..., because ...
# WORDS WE EXPECT TO MATTER: ..., ..., ...   /   ..., ..., ...

# Look at the data first. Print a few rows from each pile, at full length, and read
# two of them aloud. Anything you can only learn by looking, you learn here.

## Station 2 · Features: turn text into numbers

A logistic regression cannot see text. Something has to turn each document into a row of
numbers, and that something is your first modelling choice.

Build the matrix: rows are documents, columns are words, cells are counts. Then answer, in a
comment: how many columns did you get, and what did the vectorizer's defaults silently decide
for you (lowercasing, punctuation, what counts as a word, whether *run* and *running* are one
column or two)?

In [ ]:
# Turn df["text"] into a matrix X, and df["label"] into a 0/1 target y.
# CountVectorizer and TfidfVectorizer are both imported above.
#
# Then print the shape, and the first 20 feature names.
# ANSWER IN A COMMENT: how many columns, and which defaults made that number?

## Station 3 · Fit the model

Split off a quarter of the rows the model never sees, fit `LogisticRegression` on the rest,
and score it on the held-out quarter.

Say what fitting *did*, in one sentence, before you run it: the model is looking for one weight
per word such that the weighted sum of a document's words leans the right way for as many
training documents as possible.

If you get a convergence warning, that is the optimiser saying it ran out of iterations, not
that the model is wrong. Ask the AI what `max_iter` does.

In [ ]:
# Split, fit, score. Keep the split reproducible (random_state=0) so your two
# runs at Station 7 are comparable.

## Station 4 · Judge it honestly

An accuracy number alone means nothing. Three things make it mean something:

1. **A baseline.** What does a model that always guesses the bigger pile get? `DummyClassifier`
   will tell you. Your model's score is only interesting as a distance above that.
2. **The held-out rule.** Score on rows the model never trained on. Score on the training rows
   and you are asking whether it memorised, which it did.
3. **The shape of the errors.** A confusion matrix says *which* side it gets wrong. A model that
   is 60 percent accurate by calling almost everything pile A is a different animal from one
   that is 60 percent accurate on both.

Optional, if your group is quick: `cross_val_score` refits on five different splits, so you can
see how much your one accuracy number wobbles depending on which quarter got held out.

In [ ]:
# Baseline, held-out accuracy, confusion matrix. Then, in a comment:
# HOW MUCH BETTER THAN THE BASELINE: ...
# WHICH SIDE IT GETS WRONG: ...

## Station 5 · Read its mind

The reason today's model is a logistic regression and not something cleverer: you can read it.
Each word has one signed weight. Positive weights push toward one pile, negative toward the
other, and the largest of each are the model's reasoning laid out in full.

Pull the top ten words on each side, print them, and plot them if you have time. Then do the
part that is actually the lesson: **sort the words into topic, register, and community habit**.
*Beaches* is topic. *Citation needed* is register. *This sub* and *mods* are habit. Which kind
is your model living on, and does that answer the question you wrote at Station 1?

In [ ]:
# Top ten words per side, from the fitted model's coefficients.
# TOPIC / REGISTER / HABIT, our sort: ...

## Station 6 · Break it

Every classifier here has exactly two boxes and no concept of *neither*. Hand it a line of
Shakespeare, a recipe, a sentence in another language: it will answer, confidently, with one
of your two labels and a probability.

Feed it three inputs it has no business classifying and record what it says. Then find a real
document it gets wrong, read that document, and say why. The Pynchon misread in Underwood's
genre classifier is exactly this move done well: the error is what shows you where the category
is fuzzy.

In [ ]:
# A predict() helper that takes a string and prints the label and probability.
# Then: three out-of-domain inputs, and one real document it gets wrong.
# WHY IT FAILED: ...

## Station 7 · Change one thing, fit it again (this is the modelling)

One model is a result; several models are an argument. Fit at least three more, changing exactly
one decision at a time, and keep the numbers side by side. The candidates:

| Decision | What you change | What it should do |
|---|---|---|
| Rare words | `min_df=3` (a word must appear in 3+ documents) | drops thousands of one-off columns |
| Weighting | `TfidfVectorizer` instead of `CountVectorizer` | discounts words that are everywhere |
| Word pairs | `ngram_range=(1, 2)` | lets *this sub* be one feature |
| Regularisation | `C=0.1` vs `C=10` | how hard the model is pushed toward small weights |
| Imbalance | `class_weight="balanced"` | stops it from riding the bigger pile |

Two questions to answer with the table, not with opinion: **does the accuracy actually move**,
and **do the top weights change more than the accuracy does?** A change that leaves the score
alone but rewrites the word lists is telling you the score was never the finding.

In [ ]:
# Build a small results table: one row per model, with the choice you changed, the
# held-out accuracy, and its top three words per side. Refit, append, print.
#
# results = []
# ... for each variant: fit, then results.append({...})
# pd.DataFrame(results)

## Station 8 · Report back (fill this in, five minutes before the end)

Copy this into your own copy and fill the blanks. Each group reads theirs out; the ten answers
together are the actual finding of the session.

- **Our two piles:** \_\_\_ vs \_\_\_ , chosen because \_\_\_
- **Baseline / our best model:** \_\_\_ % / \_\_\_ %
- **The model we kept, and the one decision that made it best:** \_\_\_
- **Top words, side A:** \_\_\_  **side B:** \_\_\_
- **Topic, register, or habit?** \_\_\_
- **Where it broke:** \_\_\_
- **One caveat we would put in writing:** \_\_\_
- **What we would need to believe this:** \_\_\_

In [ ]:
# Optional: save today's work so it survives the runtime.
# The results table as a CSV in PROJECT_DIR, and a plot of the top weights as a PNG.

---

### If you are stuck

- Ask the AI for the piece, not the notebook: *"vectorize df['text'] with CountVectorizer and
  show me the shape"* beats *"do week 3 for me."* Then read what it wrote before you run it.
- Predict, run, interrogate. Every cell, all term.
- The worked version of this exact pipeline is `week03_classification.ipynb`. Peeking is
  allowed, and copying without reading is how you end up unable to defend the number.
- Errors: `../kits/common-errors-cheatsheet.md`.

### Homework it feeds

Your sketch is one of today's models on a labelled set *you* are curious about, plus a
screenshot of its five most positive and five most negative words. And bring your corpus
existence proof to Week 4: a screenshot of 50 loadable rows of the data you want to use.